In [ ]:
"""
goatpy_napari_gui.py
====================
Napari GUI for goatpy spatial glycomics analysis.

Features
--------
1. Selecting a glycan renders it as a napari Image layer ON the H&E viewer,
   coloured with the inferno colourmap.  No more sidebar scatter plot for
   spatial maps.

2. Spectrum widget (bottom dock) loads the FULL raw spectrum directly from
   the imzML file stored in sdata["maldi_adata"].uns["maldi_path"].
   Curated peaks that are present in the sdata object are highlighted in red.

3. Interactive spectrum: scroll to pan left/right, Ctrl+scroll or pinch to
   zoom, left-click on any curated peak line to select that glycan and
   immediately render it on the H&E viewer.

4. Analysis sidebar tabs: Violin/Box by cluster, Histogram, UMAP, Heatmap,
   Stats — unchanged from v2.
"""

from __future__ import annotations

import warnings
import threading
from typing import Optional, Callable

import numpy as np
import pandas as pd

# ── Qt ───────────────────────────────────────────────────────────────────────
from qtpy.QtWidgets import (
    QWidget, QVBoxLayout, QHBoxLayout, QLabel, QComboBox,
    QPushButton, QGroupBox, QSizePolicy, QScrollArea, QTabWidget,
    QCheckBox, QSpinBox, QDoubleSpinBox, QApplication, QProgressBar,
)
from qtpy.QtCore import Qt, Signal, QTimer, QThread, QObject
from qtpy.QtGui import QCursor

# ── matplotlib ───────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Qt5Agg")
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure
from matplotlib.backend_bases import MouseButton
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

# ── napari ───────────────────────────────────────────────────────────────────
import napari
from napari.utils.notifications import show_info

# ── spatialdata ──────────────────────────────────────────────────────────────
from spatialdata import SpatialData


# ════════════════════════════════════════════════════════════════════════════
# Colour palette
# ════════════════════════════════════════════════════════════════════════════

PALETTE = {
    "bg":          "#1e1e2e",
    "surface":     "#2a2a3e",
    "border":      "#3d3d5c",
    "accent":      "#7c6af7",
    "accent2":     "#f5a623",
    "text":        "#cdd6f4",
    "text_dim":    "#6c7086",
    "spectrum":    "#7c6af7",
    "raw_spec":    "#4a5580",       # dimmer colour for the dense raw spectrum
    "peak_marker": "#f38ba8",
    "highlight":   "#fab387",
    "success":     "#a6e3a1",
}

BASE_STYLE = f"""
    QWidget {{
        background-color: {PALETTE['bg']};
        color: {PALETTE['text']};
        font-family: 'Segoe UI', 'SF Pro Display', sans-serif;
        font-size: 11px;
    }}
    QGroupBox {{
        background-color: {PALETTE['surface']};
        border: 1px solid {PALETTE['border']};
        border-radius: 6px;
        margin-top: 8px;
        padding-top: 8px;
        font-weight: bold;
    }}
    QGroupBox::title {{
        subcontrol-origin: margin;
        left: 8px;
        padding: 0 4px;
        color: {PALETTE['accent']};
    }}
    QComboBox {{
        background-color: {PALETTE['surface']};
        border: 1px solid {PALETTE['border']};
        border-radius: 4px;
        padding: 4px 8px;
        color: {PALETTE['text']};
    }}
    QComboBox::drop-down {{ border: none; }}
    QComboBox QAbstractItemView {{
        background-color: {PALETTE['surface']};
        color: {PALETTE['text']};
        selection-background-color: {PALETTE['accent']};
    }}
    QPushButton {{
        background-color: {PALETTE['accent']};
        color: white;
        border: none;
        border-radius: 4px;
        padding: 6px 12px;
        font-weight: bold;
    }}
    QPushButton:hover {{ background-color: #9b8df8; }}
    QPushButton:pressed {{ background-color: #5d4ed6; }}
    QPushButton:disabled {{
        background-color: {PALETTE['border']};
        color: {PALETTE['text_dim']};
    }}
    QLabel {{ color: {PALETTE['text']}; }}
    QTabWidget::pane {{
        border: 1px solid {PALETTE['border']};
        background-color: {PALETTE['surface']};
        border-radius: 4px;
    }}
    QTabBar::tab {{
        background-color: {PALETTE['bg']};
        color: {PALETTE['text_dim']};
        padding: 6px 14px;
        border: 1px solid {PALETTE['border']};
        border-bottom: none;
        border-top-left-radius: 4px;
        border-top-right-radius: 4px;
    }}
    QTabBar::tab:selected {{
        background-color: {PALETTE['surface']};
        color: {PALETTE['text']};
    }}
    QCheckBox {{ color: {PALETTE['text']}; spacing: 6px; }}
    QScrollArea {{ border: none; background-color: transparent; }}
    QSpinBox, QDoubleSpinBox {{
        background-color: {PALETTE['surface']};
        border: 1px solid {PALETTE['border']};
        border-radius: 4px;
        padding: 3px 6px;
        color: {PALETTE['text']};
    }}
    QProgressBar {{
        background-color: {PALETTE['surface']};
        border: 1px solid {PALETTE['border']};
        border-radius: 3px;
        text-align: center;
        color: {PALETTE['text']};
    }}
    QProgressBar::chunk {{
        background-color: {PALETTE['accent']};
        border-radius: 3px;
    }}
"""


# ════════════════════════════════════════════════════════════════════════════
# m/z resolution helpers
# ════════════════════════════════════════════════════════════════════════════

def _resolve_mz_array(adata) -> np.ndarray:
    """
    Return float64 m/z array for every adata column, handling:
      1. adata.var["mz_original"]  (set by annotate_glycans)
      2. "mz-933.4" prefixed var_names
      3. plain numeric var_names
      4. column index fallback
    """
    n = adata.n_vars
    if "mz_original" in adata.var.columns:
        try:
            arr = pd.to_numeric(adata.var["mz_original"], errors="coerce").values
            if not np.any(np.isnan(arr)):
                return arr.astype(np.float64)
        except Exception:
            pass

    mzs = np.full(n, np.nan, dtype=np.float64)
    for i, vn in enumerate(adata.var_names):
        s = str(vn).strip()
        if s.lower().startswith("mz-"):
            s = s[3:]
        try:
            mzs[i] = float(s)
        except ValueError:
            pass
    nan_mask = np.isnan(mzs)
    if nan_mask.any():
        mzs[nan_mask] = np.where(nan_mask)[0].astype(np.float64)
    return mzs


def _resolve_var_display_labels(adata) -> list[str]:
    mzs = _resolve_mz_array(adata)
    labels = []
    for i, vn in enumerate(adata.var_names):
        s = str(vn).strip()
        labels.append(f"{mzs[i]:.4f}" if s.lower().startswith("mz-") else s)
    return labels


def _looks_numeric(s: str) -> bool:
    s2 = s.strip()
    if s2.lower().startswith("mz-"):
        s2 = s2[3:]
    try:
        float(s2)
        return True
    except ValueError:
        return False


# ════════════════════════════════════════════════════════════════════════════
# Napari layer helper — render glycan ion image ON the H&E
# ════════════════════════════════════════════════════════════════════════════

GLYCAN_LAYER_NAME = "glycan_ion_map"

from spatialdata import transform
from napari.utils.notifications import show_info
import numpy as np
from spatialdata import transform
from napari.utils.notifications import show_info
import numpy as np


def _render_glycan_on_viewer(
    viewer,
    sdata,
    peak_mz,
    label,
    table_name="maldi_adata",
):
    """
    Render glycan intensity directly onto the aligned MALDI pixel polygons
    stored in sdata.shapes["pixels"].

    This avoids rasterisation and uses the registered SpatialData geometry
    as the source of truth.
    """

    adata = sdata.tables[table_name]

    # ------------------------------------------------------------
    # Find requested m/z
    # ------------------------------------------------------------

    var_mzs = _resolve_mz_array(adata)

    idx = int(np.argmin(np.abs(var_mzs - peak_mz)))

    if abs(var_mzs[idx] - peak_mz) > 0.5:
        show_info(
            f"Peak {peak_mz:.2f} not found "
            f"(closest: {var_mzs[idx]:.2f})"
        )
        return

    values = np.asarray(
        adata.X[:, idx]
    ).astype(np.float32).flatten()

    # ------------------------------------------------------------
    # Get aligned MALDI pixel polygons
    # ------------------------------------------------------------

    try:

        pixels = transform(
            sdata.shapes["pixels"],
            to_coordinate_system="aligned",
        ).copy()

    except Exception as e:

        show_info(
            f"Could not transform pixel geometries: {e}"
        )
        return

    # ------------------------------------------------------------
    # Sanity check
    # ------------------------------------------------------------

    if len(pixels) != len(values):

        show_info(
            f"Pixel count mismatch: "
            f"{len(pixels)} polygons vs "
            f"{len(values)} intensities"
        )
        return

    pixels["intensity"] = values

    # ------------------------------------------------------------
    # Convert polygons into napari format
    # ------------------------------------------------------------

    shape_data = []
    intensity_values = []

    for intensity, geom in zip(
        pixels["intensity"].values,
        pixels.geometry,
    ):

        if geom.is_empty:
            continue

        if geom.geom_type == "Polygon":

            coords = np.asarray(
                geom.exterior.coords
            )

            # Napari expects Y,X ordering
            shape_data.append(
                coords[:, [1, 0]]
            )

            intensity_values.append(
                float(intensity)
            )

        elif geom.geom_type == "MultiPolygon":

            largest = max(
                geom.geoms,
                key=lambda g: g.area
            )

            coords = np.asarray(
                largest.exterior.coords
            )

            shape_data.append(
                coords[:, [1, 0]]
            )

            intensity_values.append(
                float(intensity)
            )

    if len(shape_data) == 0:

        show_info(
            "No valid MALDI geometries found."
        )
        return

    # ------------------------------------------------------------
    # Contrast limits
    # ------------------------------------------------------------

    nonzero = np.asarray(
        intensity_values
    )

    nonzero = nonzero[
        nonzero > 0
    ]

    if len(nonzero):

        cmin = float(
            np.percentile(nonzero, 1)
        )

        cmax = float(
            np.percentile(nonzero, 99)
        )

    else:

        cmin = 0.0
        cmax = 1.0

    # ------------------------------------------------------------
    # Layer naming
    # ------------------------------------------------------------

    short = (
        label.split("(")[0]
        .strip()[:25]
    )

    layer_name = (
        f"{GLYCAN_LAYER_NAME} [{short}]"
    )

    # ------------------------------------------------------------
    # Remove previous glycan layer
    # ------------------------------------------------------------

    for layer in list(viewer.layers):

        if layer.name.startswith(
            GLYCAN_LAYER_NAME
        ):
            viewer.layers.remove(layer)

    # ------------------------------------------------------------
    # Add polygon layer
    # ------------------------------------------------------------

    viewer.add_shapes(
        shape_data,
        shape_type="polygon",
        properties={
            "intensity": np.asarray(
                intensity_values,
                dtype=np.float32,
            )
        },
        face_color="intensity",
        face_colormap="inferno",
        edge_width=0,
        opacity=0.75,
        blending="additive",
        name=layer_name,
    )

    layer = viewer.layers[layer_name]

    # Apply contrast scaling
    layer.face_contrast_limits = (
        cmin,
        cmax,
    )

    show_info(
        f"Ion map: {short} "
        f"({peak_mz:.2f} Da)"
    )
    
    
# ════════════════════════════════════════════════════════════════════════════
# Background worker for loading full imzML spectrum
# ════════════════════════════════════════════════════════════════════════════

class _SpectrumLoader(QObject):
    """
    Loads the mean spectrum from the raw imzML file on a worker thread,
    then emits finished(mz_array, intensity_array).
    """
    finished = Signal(object, object)   # np.ndarray, np.ndarray
    progress = Signal(int)              # 0-100

    def __init__(self, imzml_path: str, n_sample: int = 500):
        super().__init__()
        self.imzml_path = imzml_path
        self.n_sample = n_sample        # max spectra to average (for speed)

    def run(self):
        try:
            from pyimzml.ImzMLParser import ImzMLParser
            p = ImzMLParser(self.imzml_path)
            n_total = len(p.coordinates)
            step = max(1, n_total // self.n_sample)
            indices = list(range(0, n_total, step))

            # First pass: discover global m/z range
            all_mzs = []
            for i, idx in enumerate(indices):
                mzs, _ = p.getspectrum(idx)
                all_mzs.append(mzs)
                if i % max(1, len(indices) // 20) == 0:
                    self.progress.emit(int(i / len(indices) * 50))

            lo = min(a[0] for a in all_mzs if len(a))
            hi = max(a[-1] for a in all_mzs if len(a))
            # Build uniform 0.05 Da grid
            bin_edges = np.arange(lo, hi + 0.05, 0.05)
            acc = np.zeros(len(bin_edges) - 1, dtype=np.float64)
            counts = np.zeros(len(bin_edges) - 1, dtype=np.int32)

            for i, (idx, mzs) in enumerate(zip(indices, all_mzs)):
                _, ints = p.getspectrum(idx)
                if len(mzs) == 0:
                    continue
                bin_idx = np.searchsorted(bin_edges, mzs, side="right") - 1
                valid = (bin_idx >= 0) & (bin_idx < len(acc))
                np.add.at(acc, bin_idx[valid], np.asarray(ints, dtype=np.float64)[valid])
                np.add.at(counts, bin_idx[valid], 1)
                if i % max(1, len(indices) // 20) == 0:
                    self.progress.emit(50 + int(i / len(indices) * 50))

            with np.errstate(divide="ignore", invalid="ignore"):
                mean_spec = np.where(counts > 0, acc / counts, 0.0)

            bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2.0
            self.progress.emit(100)
            self.finished.emit(bin_centres, mean_spec)

        except Exception as e:
            print(f"[SpectrumLoader] Error: {e}")
            self.finished.emit(np.array([]), np.array([]))


# ════════════════════════════════════════════════════════════════════════════
# MplCanvas
# ════════════════════════════════════════════════════════════════════════════

class MplCanvas(FigureCanvas):
    def __init__(self, parent=None, width=8, height=3, dpi=90):
        self.fig = Figure(figsize=(width, height), dpi=dpi)
        self.fig.patch.set_facecolor(PALETTE["surface"])
        super().__init__(self.fig)
        self.setParent(parent)
        self.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.setMinimumHeight(180)

    def _style_ax(self, ax):
        ax.set_facecolor(PALETTE["bg"])
        for spine in ax.spines.values():
            spine.set_color(PALETTE["border"])
        ax.tick_params(colors=PALETTE["text_dim"], labelsize=8)
        ax.xaxis.label.set_color(PALETTE["text"])
        ax.yaxis.label.set_color(PALETTE["text"])
        ax.title.set_color(PALETTE["text"])
        return ax


# ════════════════════════════════════════════════════════════════════════════
# 1. SPECTRUM WIDGET  (bottom dock)
#    • Loads full raw spectrum from imzML in background
#    • Scroll = pan, Ctrl+scroll = zoom
#    • Click near a red peak line → selects that glycan
# ════════════════════════════════════════════════════════════════════════════

class SpectrumWidget(QWidget):
    """Interactive spectrum panel — full imzML background + clickable peaks."""

    # Emitted when user clicks a peak; consumed by AnalysisSidebar + viewer
    peak_clicked = Signal(float, str)   # mz, display_label

    def __init__(
        self,
        sdata: SpatialData,
        peaks: list[float],
        glycan_df: Optional[pd.DataFrame] = None,
        table_name: str = "maldi_adata",
        parent=None,
    ):
        super().__init__(parent)
        self.sdata = sdata
        self.peaks = sorted(peaks)
        self.glycan_df = glycan_df
        self.table_name = table_name

        self.highlighted_mz: Optional[float] = None
        self.highlighted_label: str = ""
        self._tol = 0.15

        # Raw full spectrum (from imzML)
        self._raw_mz: Optional[np.ndarray] = None
        self._raw_int: Optional[np.ndarray] = None

        # sdata mean spectrum (fallback / overlay reference)
        self._sdata_mz: Optional[np.ndarray] = None
        self._sdata_int: Optional[np.ndarray] = None

        # View window
        self._view_lo: float = 500.0
        self._view_hi: float = 3000.0

        # Peak label lookup mz → display string
        self._peak_labels: dict[float, str] = {}
        self._build_peak_labels()

        self._build_ui()
        QTimer.singleShot(100, self._load_sdata_spectrum)
        QTimer.singleShot(200, self._start_imzml_load)

    # ── Peak label lookup ─────────────────────────────────────────────────

    def _build_peak_labels(self):
        """Map each curated m/z to its best glycan name."""
        try:
            adata = self.sdata.tables[self.table_name]
            mz_arr = _resolve_mz_array(adata)
            disp = _resolve_var_display_labels(adata)
            for mz, lbl in zip(mz_arr, disp):
                self._peak_labels[float(mz)] = lbl
        except Exception:
            pass

        if self.glycan_df is not None:
            for _, row in self.glycan_df.iterrows():
                mz = float(row["mz"])
                lbl = str(row["label"])
                if lbl and lbl not in ("nan", ""):
                    self._peak_labels[mz] = lbl

    def _label_for_peak(self, mz: float) -> str:
        best = min(self._peak_labels.keys(), key=lambda m: abs(m - mz), default=None)
        if best is not None and abs(best - mz) < 0.5:
            return self._peak_labels[best]
        return f"{mz:.4f}"

    # ── UI ────────────────────────────────────────────────────────────────

    def _build_ui(self):
        layout = QVBoxLayout(self)
        layout.setContentsMargins(4, 4, 4, 4)
        layout.setSpacing(2)

        # ── Controls bar ─────────────────────────────────────────────────
        ctrl = QHBoxLayout()

        lbl = QLabel("Spectrum source:")
        lbl.setFixedWidth(110)
        ctrl.addWidget(lbl)
        self.source_combo = QComboBox()
        self.source_combo.addItems(["Raw imzML (full)", "sdata mean"])
        self.source_combo.setFixedWidth(160)
        self.source_combo.currentTextChanged.connect(self._redraw)
        ctrl.addWidget(self.source_combo)

        ctrl.addSpacing(16)
        ctrl.addWidget(QLabel("Show peaks:"))
        self.show_peaks_cb = QCheckBox()
        self.show_peaks_cb.setChecked(True)
        self.show_peaks_cb.stateChanged.connect(self._redraw)
        ctrl.addWidget(self.show_peaks_cb)

        ctrl.addSpacing(16)
        ctrl.addWidget(QLabel("Zoom to:"))
        self.mz_lo = QDoubleSpinBox()
        self.mz_lo.setRange(50, 10000)
        self.mz_lo.setValue(500)
        self.mz_lo.setSingleStep(50)
        self.mz_lo.setFixedWidth(78)
        ctrl.addWidget(self.mz_lo)
        ctrl.addWidget(QLabel("–"))
        self.mz_hi = QDoubleSpinBox()
        self.mz_hi.setRange(50, 10000)
        self.mz_hi.setValue(3000)
        self.mz_hi.setSingleStep(50)
        self.mz_hi.setFixedWidth(78)
        ctrl.addWidget(self.mz_hi)
        zoom_btn = QPushButton("Go")
        zoom_btn.setFixedWidth(40)
        zoom_btn.clicked.connect(self._apply_zoom)
        ctrl.addWidget(zoom_btn)

        reset_btn = QPushButton("Reset")
        reset_btn.setFixedWidth(55)
        reset_btn.clicked.connect(self._reset_zoom)
        ctrl.addWidget(reset_btn)

        ctrl.addStretch()
        self.status_lbl = QLabel("Loading…")
        self.status_lbl.setStyleSheet(f"color: {PALETTE['text_dim']};")
        ctrl.addWidget(self.status_lbl)

        layout.addLayout(ctrl)

        # ── Progress bar (shown while imzML loads) ────────────────────────
        self.progress_bar = QProgressBar()
        self.progress_bar.setFixedHeight(4)
        self.progress_bar.setTextVisible(False)
        self.progress_bar.hide()
        layout.addWidget(self.progress_bar)

        # ── Hint label ────────────────────────────────────────────────────
        hint = QLabel(
            "Scroll to pan  ·  Ctrl+scroll to zoom  ·  Click a red peak line to select glycan"
        )
        hint.setStyleSheet(f"color: {PALETTE['text_dim']}; font-size: 9px;")
        layout.addWidget(hint)

        # ── Canvas ────────────────────────────────────────────────────────
        self.canvas = MplCanvas(self, width=10, height=2.6, dpi=90)
        layout.addWidget(self.canvas)

        # ── Interactions ──────────────────────────────────────────────────
        self.canvas.mpl_connect("scroll_event",         self._on_scroll)
        self.canvas.mpl_connect("button_press_event",   self._on_click)

    # ── Data loading ──────────────────────────────────────────────────────

    def _load_sdata_spectrum(self):
        try:
            adata = self.sdata.tables[self.table_name]
            mz_arr = _resolve_mz_array(adata)
            X = np.asarray(adata.X, dtype=np.float32)
            intensities = X.mean(axis=0).astype(np.float64)
            self._sdata_mz = mz_arr
            self._sdata_int = intensities
            self._view_lo = float(mz_arr.min())
            self._view_hi = float(mz_arr.max())
            self.mz_lo.setValue(self._view_lo)
            self.mz_hi.setValue(self._view_hi)
            self.status_lbl.setText(
                f"{adata.n_obs:,} pixels · {adata.n_vars:,} peaks  |  "
                "Loading full spectrum…"
            )
        except Exception as e:
            self.status_lbl.setText(f"sdata load error: {e}")
        self._redraw()

    def _start_imzml_load(self):
        try:
            path = self.sdata.tables[self.table_name].uns.get("maldi_path")
        except Exception:
            path = None

        if not path:
            self.status_lbl.setText(
                self.status_lbl.text().replace("Loading full spectrum…", "No imzML path found")
            )
            return

        self.progress_bar.show()
        self.progress_bar.setValue(0)

        self._loader = _SpectrumLoader(path, n_sample=800)
        self._thread = QThread()
        self._loader.moveToThread(self._thread)
        self._thread.started.connect(self._loader.run)
        self._loader.finished.connect(self._on_imzml_loaded)
        self._loader.progress.connect(self.progress_bar.setValue)
        self._loader.finished.connect(self._thread.quit)
        self._thread.start()

    def _on_imzml_loaded(self, mz_arr, int_arr):
        self.progress_bar.hide()
        if len(mz_arr) == 0:
            self.status_lbl.setText("imzML load failed — using sdata mean")
            return

        self._raw_mz = mz_arr
        self._raw_int = int_arr

        # Switch to raw source automatically
        self.source_combo.setCurrentText("Raw imzML (full)")

        # Update view window to full range
        self._view_lo = float(mz_arr.min())
        self._view_hi = float(mz_arr.max())
        self.mz_lo.setValue(self._view_lo)
        self.mz_hi.setValue(self._view_hi)

        try:
            adata = self.sdata.tables[self.table_name]
            self.status_lbl.setText(
                f"{adata.n_obs:,} pixels · {adata.n_vars:,} peaks  |  "
                f"Full spectrum: {len(mz_arr):,} bins"
            )
        except Exception:
            pass

        self._redraw()

    # ── Interactions ──────────────────────────────────────────────────────

    def _on_scroll(self, event):
        """
        Plain scroll  → pan (shift view left/right).
        Ctrl+scroll   → zoom in/out around cursor position.
        """
        if self._current_mz() is None:
            return

        span = self._view_hi - self._view_lo
        if span <= 0:
            return

        ctrl_held = (event.key == "control") or (
            QApplication.keyboardModifiers() & Qt.ControlModifier
        )

        if ctrl_held:
            # Zoom: shrink/expand around cursor
            factor = 0.85 if event.button == "up" else 1.0 / 0.85
            cursor_mz = event.xdata if event.xdata is not None else (self._view_lo + self._view_hi) / 2
            new_lo = cursor_mz - (cursor_mz - self._view_lo) * factor
            new_hi = cursor_mz + (self._view_hi - cursor_mz) * factor
        else:
            # Pan: move 15% of span per scroll tick
            shift = span * 0.15 * (-1 if event.button == "up" else 1)
            new_lo = self._view_lo + shift
            new_hi = self._view_hi + shift

        # Clamp to data extent
        mz = self._current_mz()
        data_lo, data_hi = float(mz.min()), float(mz.max())
        width = new_hi - new_lo
        new_lo = max(data_lo, new_lo)
        new_hi = min(data_hi, new_hi)
        if new_hi - new_lo < 5:
            new_lo = new_hi - 5

        self._view_lo = new_lo
        self._view_hi = new_hi
        self.mz_lo.setValue(new_lo)
        self.mz_hi.setValue(new_hi)
        self._redraw()

    def _on_click(self, event):
        """Left-click: find nearest curated peak within a tolerance and select it."""
        if event.button != MouseButton.LEFT:
            return
        if event.xdata is None:
            return
        if not self.show_peaks_cb.isChecked():
            return

        click_mz = float(event.xdata)
        span = self._view_hi - self._view_lo
        snap_tol = span * 0.015          # 1.5% of visible window

        # Find nearest curated peak within snap_tol
        visible_peaks = [p for p in self.peaks if self._view_lo <= p <= self._view_hi]
        if not visible_peaks:
            return

        nearest = min(visible_peaks, key=lambda p: abs(p - click_mz))
        if abs(nearest - click_mz) > snap_tol:
            return

        label = self._label_for_peak(nearest)
        self.highlight_glycan(nearest, label)
        self.peak_clicked.emit(nearest, label)

    # ── View helpers ──────────────────────────────────────────────────────

    def _current_mz(self) -> Optional[np.ndarray]:
        src = self.source_combo.currentText()
        if src == "Raw imzML (full)" and self._raw_mz is not None:
            return self._raw_mz
        return self._sdata_mz

    def _apply_zoom(self):
        self._view_lo = self.mz_lo.value()
        self._view_hi = self.mz_hi.value()
        self._redraw()

    def _reset_zoom(self):
        mz = self._current_mz()
        if mz is not None:
            self._view_lo = float(mz.min())
            self._view_hi = float(mz.max())
            self.mz_lo.setValue(self._view_lo)
            self.mz_hi.setValue(self._view_hi)
        self._redraw()

    # ── Drawing ──────────────────────────────────────────────────────────

    def _redraw(self):
        src = self.source_combo.currentText()
        if src == "Raw imzML (full)" and self._raw_mz is not None:
            mz, intensity = self._raw_mz, self._raw_int
        elif self._sdata_mz is not None:
            mz, intensity = self._sdata_mz, self._sdata_int
        else:
            return

        lo, hi = self._view_lo, self._view_hi
        mask = (mz >= lo) & (mz <= hi)
        mz_v = mz[mask]
        int_v = intensity[mask]
        if len(mz_v) == 0:
            return

        self.canvas.fig.clear()
        ax = self.canvas.fig.add_subplot(111)
        self.canvas._style_ax(ax)

        # ── Background spectrum ───────────────────────────────────────────
        spec_col = PALETTE["raw_spec"] if src == "Raw imzML (full)" else PALETTE["spectrum"]
        ax.plot(mz_v, int_v, color=spec_col, linewidth=0.6, alpha=0.8, zorder=2)
        ax.fill_between(mz_v, int_v, color=spec_col, alpha=0.08, zorder=1)
        max_int = float(int_v.max()) if len(int_v) else 1.0

        # ── Curated peaks (red dashed lines) ─────────────────────────────
        if self.show_peaks_cb.isChecked():
            for pk in self.peaks:
                if lo <= pk <= hi:
                    ax.axvline(
                        pk, color=PALETTE["peak_marker"],
                        linewidth=1.0, linestyle="--", alpha=0.7, zorder=3,
                        picker=5,
                    )

        # ── Highlighted / selected glycan ─────────────────────────────────
        if self.highlighted_mz is not None:
            hmz = self.highlighted_mz
            if lo <= hmz <= hi:
                ax.axvspan(hmz - self._tol, hmz + self._tol,
                           color=PALETTE["highlight"], alpha=0.22, zorder=4)
                ax.axvline(hmz, color=PALETTE["highlight"],
                           linewidth=1.8, linestyle="-", alpha=0.95, zorder=5)
                ax.text(
                    hmz, max_int * 1.01, self.highlighted_label,
                    color=PALETTE["highlight"], fontsize=7.5,
                    ha="center", va="bottom", rotation=90, clip_on=True,
                )

        # ── Legend ────────────────────────────────────────────────────────
        handles = [
            Line2D([0], [0], color=spec_col, linewidth=1.5,
                   label="Full spectrum" if src == "Raw imzML (full)" else "sdata mean"),
        ]
        if self.show_peaks_cb.isChecked():
            handles.append(
                Line2D([0], [0], color=PALETTE["peak_marker"], linewidth=1,
                       linestyle="--", label=f"Curated peaks (n={len(self.peaks)})")
            )
        if self.highlighted_mz is not None:
            handles.append(
                mpatches.Patch(color=PALETTE["highlight"], alpha=0.5,
                               label=f"Selected: {self.highlighted_label}")
            )
        ax.legend(handles=handles, fontsize=7.5, framealpha=0.3,
                  loc="upper right",
                  facecolor=PALETTE["surface"], labelcolor=PALETTE["text"])

        ax.set_xlabel("m/z (Da)", fontsize=9)
        ax.set_ylabel("Intensity", fontsize=9)
        ax.set_xlim(lo, hi)
        ax.set_ylim(bottom=0)
        ax.xaxis.set_minor_locator(mticker.AutoMinorLocator(5))
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        self.canvas.fig.tight_layout(pad=0.4)
        self.canvas.draw()

    # ── Public API ────────────────────────────────────────────────────────

    def highlight_glycan(self, mz: float, label: str, tol: float = 0.15):
        """Highlight a glycan in the spectrum and scroll it into view."""
        self.highlighted_mz = mz
        self.highlighted_label = label
        self._tol = tol

        # Centre the view on this peak, keeping current span or defaulting to 400 Da
        span = max(self._view_hi - self._view_lo, 50)
        half = span / 2
        mz_full = self._current_mz()
        if mz_full is not None:
            data_lo, data_hi = float(mz_full.min()), float(mz_full.max())
            new_lo = max(data_lo, mz - half)
            new_hi = min(data_hi, mz + half)
            self._view_lo = new_lo
            self._view_hi = new_hi
            self.mz_lo.setValue(new_lo)
            self.mz_hi.setValue(new_hi)

        self._redraw()

    def clear_highlight(self):
        self.highlighted_mz = None
        self.highlighted_label = ""
        self._redraw()


# ════════════════════════════════════════════════════════════════════════════
# 2. ANALYSIS SIDEBAR  (right dock)
# ════════════════════════════════════════════════════════════════════════════

class AnalysisSidebar(QWidget):
    """
    Right-dock panel.
    Glycan tab: selector + Violin/Box/Histogram (no spatial scatter — that is
    now rendered on the napari viewer directly).
    """

    glycan_selected = Signal(float, str)   # mz, display_label

    def __init__(
        self,
        sdata: SpatialData,
        peaks: list[float],
        viewer: napari.Viewer,
        glycan_df: Optional[pd.DataFrame] = None,
        table_name: str = "maldi_adata",
        parent=None,
    ):
        super().__init__(parent)
        self.sdata = sdata
        self.peaks = sorted(peaks)
        self.viewer = viewer
        self.glycan_df = glycan_df
        self.table_name = table_name

        self._build_glycan_lookup()
        self._build_ui()

    # ── Data helpers ──────────────────────────────────────────────────────

    def _build_glycan_lookup(self):
        self.mz_to_label: dict[float, str] = {}
        self.label_to_mz: dict[str, float] = {}
        self.glycan_names: list[str] = []

        try:
            adata = self.sdata.tables[self.table_name]
            mz_arr = _resolve_mz_array(adata)
            disp = _resolve_var_display_labels(adata)
            for mz, lbl in zip(mz_arr, disp):
                self.mz_to_label[mz] = lbl
                self.label_to_mz[lbl] = mz
        except Exception:
            pass

        if self.glycan_df is not None:
            for _, row in self.glycan_df.iterrows():
                mz = float(row["mz"])
                lbl = str(row["label"])
                if lbl and lbl not in ("nan", ""):
                    self.mz_to_label[mz] = lbl
                    self.label_to_mz[lbl] = mz

        names = []
        for pk in self.peaks:
            best_label = f"{pk:.4f}"
            best_dist = float("inf")
            if self.glycan_df is not None:
                for _, row in self.glycan_df.iterrows():
                    d = abs(float(row["mz"]) - pk)
                    if d < best_dist and d < 0.5:
                        lbl = str(row["label"])
                        if lbl and lbl not in ("nan", ""):
                            best_dist = d
                            best_label = f"{lbl}  ({pk:.2f})"
            if best_dist == float("inf"):
                nearest = min(self.mz_to_label.keys(), key=lambda m: abs(m - pk), default=None)
                if nearest is not None and abs(nearest - pk) < 0.5:
                    lbl = self.mz_to_label[nearest]
                    if not _looks_numeric(lbl):
                        best_label = f"{lbl}  ({pk:.2f})"
            names.append(best_label)
        self.glycan_names = names

    def _adata(self):
        return self.sdata.tables[self.table_name]

    def _get_peak_index(self, peak_mz: float) -> Optional[int]:
        adata = self._adata()
        try:
            var_mzs = _resolve_mz_array(adata)
            idx = int(np.argmin(np.abs(var_mzs - peak_mz)))
            if abs(var_mzs[idx] - peak_mz) < 0.5:
                return idx
        except Exception:
            pass
        return None

    # ── UI ────────────────────────────────────────────────────────────────

    def _build_ui(self):
        layout = QVBoxLayout(self)
        layout.setContentsMargins(6, 6, 6, 6)
        layout.setSpacing(6)

        title = QLabel("goatpy  Analysis")
        title.setStyleSheet(
            f"font-size: 14px; font-weight: bold; color: {PALETTE['accent']}; padding: 4px 0;"
        )
        layout.addWidget(title)

        self.tabs = QTabWidget()
        layout.addWidget(self.tabs)
        self.tabs.addTab(self._build_glycan_tab(),  "Glycan")
        self.tabs.addTab(self._build_umap_tab(),    "UMAP")
        self.tabs.addTab(self._build_heatmap_tab(), "Heatmap")
        self.tabs.addTab(self._build_stats_tab(),   "Stats")

    # ── Glycan tab ────────────────────────────────────────────────────────

    def _build_glycan_tab(self) -> QWidget:
        w = QWidget()
        layout = QVBoxLayout(w)
        layout.setContentsMargins(4, 4, 4, 4)
        layout.setSpacing(6)

        # Selector
        sel_grp = QGroupBox("Select Glycan / m/z")
        sel_layout = QVBoxLayout(sel_grp)

        self.glycan_search = QComboBox()
        self.glycan_search.setEditable(True)
        self.glycan_search.setInsertPolicy(QComboBox.NoInsert)
        self.glycan_search.addItems(self.glycan_names)
        self.glycan_search.setCurrentIndex(0)
        sel_layout.addWidget(self.glycan_search)

        show_btn = QPushButton("Show on H&E Viewer")
        show_btn.clicked.connect(self._on_glycan_show)
        sel_layout.addWidget(show_btn)

        note = QLabel("Ion map is added as a napari layer over the H&E image.")
        note.setStyleSheet(f"color: {PALETTE['text_dim']}; font-size: 9px;")
        note.setWordWrap(True)
        sel_layout.addWidget(note)

        layout.addWidget(sel_grp)

        # Distribution plot
        plot_grp = QGroupBox("Distribution Plot")
        plot_layout = QVBoxLayout(plot_grp)

        row1 = QHBoxLayout()
        row1.addWidget(QLabel("Type:"))
        self.glycan_plot_type = QComboBox()
        self.glycan_plot_type.addItems(["Violin by cluster", "Box by cluster", "Histogram"])
        row1.addWidget(self.glycan_plot_type)
        plot_layout.addLayout(row1)

        row2 = QHBoxLayout()
        row2.addWidget(QLabel("Group:"))
        self.cluster_col_combo = QComboBox()
        self._populate_obs_categoricals(self.cluster_col_combo)
        row2.addWidget(self.cluster_col_combo)
        plot_layout.addLayout(row2)

        dist_btn = QPushButton("Plot Distribution")
        dist_btn.clicked.connect(self._on_dist_plot)
        plot_layout.addWidget(dist_btn)

        layout.addWidget(plot_grp)

        self.glycan_canvas = MplCanvas(w, width=4.5, height=3.8, dpi=90)
        layout.addWidget(self.glycan_canvas)

        return w

    def _populate_obs_categoricals(self, combo: QComboBox):
        try:
            obs = self._adata().obs
            cats = [c for c in obs.columns
                    if obs[c].dtype.name == "category" or
                    c in ("GPCA_clusters", "leiden", "batch", "annotation")]
            combo.clear()
            combo.addItems(cats if cats else ["(none)"])
        except Exception:
            combo.addItems(["(none)"])

    def _on_glycan_show(self):
        idx = self.glycan_search.currentIndex()
        if idx < 0 or idx >= len(self.peaks):
            return
        pk = self.peaks[idx]
        label = self.glycan_names[idx]
        self._render_on_viewer(pk, label)
        self.glycan_selected.emit(pk, label)

    def _on_dist_plot(self):
        idx = self.glycan_search.currentIndex()
        if idx < 0 or idx >= len(self.peaks):
            return
        pk = self.peaks[idx]
        label = self.glycan_names[idx]
        self._draw_distribution(pk, label)

    def _render_on_viewer(self, peak_mz: float, label: str):
        _render_glycan_on_viewer(
            self.viewer, self.sdata, peak_mz, label, self.table_name
        )

    def _draw_distribution(self, peak_mz: float, label: str):
        col_idx = self._get_peak_index(peak_mz)
        if col_idx is None:
            show_info(f"Peak {peak_mz:.2f} not found.")
            return

        adata = self._adata()
        X = np.asarray(adata.X, dtype=np.float32)
        values = X[:, col_idx]
        short = label.split("(")[0].strip()[:28]
        cluster_col = self.cluster_col_combo.currentText()
        plot_type = self.glycan_plot_type.currentText()

        self.glycan_canvas.fig.clear()
        ax = self.glycan_canvas.fig.add_subplot(111)
        self.glycan_canvas._style_ax(ax)

        if plot_type == "Histogram":
            ax.hist(values[values > 0], bins=50,
                    color=PALETTE["accent"], edgecolor="none", alpha=0.8)
            ax.set_xlabel("Intensity", fontsize=8)
            ax.set_ylabel("# Pixels", fontsize=8)
            ax.set_title(short, fontsize=9)
        else:
            self._plot_by_cluster(
                ax, adata, values, cluster_col,
                kind="violin" if "Violin" in plot_type else "box",
                label=short,
            )

        self.glycan_canvas.fig.tight_layout(pad=0.5)
        self.glycan_canvas.draw()

    def _plot_by_cluster(self, ax, adata, values, cluster_col, kind, label):
        if cluster_col == "(none)" or cluster_col not in adata.obs.columns:
            ax.text(0.5, 0.5, "No cluster column\nfound",
                    ha="center", va="center", transform=ax.transAxes,
                    color=PALETTE["text_dim"])
            return
        clusters = adata.obs[cluster_col].astype(str).values
        unique = sorted(set(clusters))
        cmap = plt.get_cmap("tab20", len(unique))
        data_by_cluster = [values[clusters == c] for c in unique]

        if kind == "violin":
            parts = ax.violinplot(data_by_cluster, positions=range(len(unique)),
                                  showmedians=True, showextrema=False)
            for i, pc in enumerate(parts["bodies"]):
                pc.set_facecolor(cmap(i)); pc.set_alpha(0.7)
            parts["cmedians"].set_color(PALETTE["text"])
        else:
            bp = ax.boxplot(data_by_cluster, positions=range(len(unique)),
                            patch_artist=True, showfliers=False,
                            medianprops={"color": PALETTE["text"], "linewidth": 1.5},
                            whiskerprops={"color": PALETTE["text_dim"]},
                            capprops={"color": PALETTE["text_dim"]})
            for i, patch in enumerate(bp["boxes"]):
                patch.set_facecolor(cmap(i)); patch.set_alpha(0.75)

        ax.set_xticks(range(len(unique)))
        ax.set_xticklabels(unique, rotation=45, ha="right", fontsize=7)
        ax.set_ylabel("Intensity", fontsize=8)
        ax.set_title(f"{label}  by {cluster_col}", fontsize=8.5)

    # ── Select glycan from spectrum click ─────────────────────────────────

    def select_peak_from_spectrum(self, mz: float, label: str):
        """Called when the user clicks a peak in the spectrum widget."""
        # Update combo box to match if possible
        for i, pk in enumerate(self.peaks):
            if abs(pk - mz) < 0.5:
                self.glycan_search.setCurrentIndex(i)
                break
        self._render_on_viewer(mz, label)

    # ── UMAP tab ──────────────────────────────────────────────────────────

    def _build_umap_tab(self) -> QWidget:
        w = QWidget()
        layout = QVBoxLayout(w)
        layout.setContentsMargins(4, 4, 4, 4)

        col_grp = QGroupBox("Colour By")
        col_layout = QVBoxLayout(col_grp)
        self.umap_color_type = QComboBox()
        self.umap_color_type.addItems(["Cluster column", "Glycan intensity"])
        self.umap_color_type.currentTextChanged.connect(self._toggle_umap_controls)
        col_layout.addWidget(self.umap_color_type)
        self.umap_cluster_col = QComboBox()
        self._populate_obs_categoricals(self.umap_cluster_col)
        col_layout.addWidget(self.umap_cluster_col)
        self.umap_glycan_combo = QComboBox()
        self.umap_glycan_combo.addItems(self.glycan_names)
        self.umap_glycan_combo.setVisible(False)
        col_layout.addWidget(self.umap_glycan_combo)
        layout.addWidget(col_grp)

        emb_grp = QGroupBox("Embedding")
        emb_layout = QHBoxLayout(emb_grp)
        self.umap_embedding_combo = QComboBox()
        self._populate_obsm(self.umap_embedding_combo)
        emb_layout.addWidget(self.umap_embedding_combo)
        layout.addWidget(emb_grp)

        plot_btn = QPushButton("Plot UMAP")
        plot_btn.clicked.connect(self._draw_umap)
        layout.addWidget(plot_btn)

        self.umap_canvas = MplCanvas(w, width=4.5, height=4.5, dpi=90)
        layout.addWidget(self.umap_canvas)
        return w

    def _toggle_umap_controls(self, text):
        is_glycan = text == "Glycan intensity"
        self.umap_glycan_combo.setVisible(is_glycan)
        self.umap_cluster_col.setVisible(not is_glycan)

    def _populate_obsm(self, combo: QComboBox):
        try:
            keys = list(self._adata().obsm.keys())
            combo.addItems(keys if keys else ["(none)"])
        except Exception:
            combo.addItems(["(none)"])

    def _draw_umap(self):
        adata = self._adata()
        emb_key = self.umap_embedding_combo.currentText()
        if emb_key == "(none)" or emb_key not in adata.obsm:
            show_info("No embedding found. Run graphpca_spatialdata() first.")
            return
        emb = adata.obsm[emb_key]
        if emb.shape[1] == 2:
            coords = emb
        else:
            try:
                from umap import UMAP
                coords = UMAP(n_components=2, random_state=42).fit_transform(emb)
            except ImportError:
                coords = emb[:, :2]

        color_type = self.umap_color_type.currentText()
        self.umap_canvas.fig.clear()
        ax = self.umap_canvas.fig.add_subplot(111)
        self.umap_canvas._style_ax(ax)

        if color_type == "Cluster column":
            col = self.umap_cluster_col.currentText()
            if col != "(none)" and col in adata.obs.columns:
                labels = adata.obs[col].astype(str).values
                unique = sorted(set(labels))
                cmap = plt.get_cmap("tab20", len(unique))
                for i, cl in enumerate(unique):
                    mask = labels == cl
                    ax.scatter(coords[mask, 0], coords[mask, 1],
                               s=1, alpha=0.6, color=cmap(i), label=cl, rasterized=True)
                ax.legend(markerscale=4, fontsize=6.5, framealpha=0.3,
                          facecolor=PALETTE["surface"], labelcolor=PALETTE["text"],
                          bbox_to_anchor=(1, 1), loc="upper left")
                ax.set_title(f"UMAP — {col}", fontsize=8.5)
            else:
                ax.scatter(coords[:, 0], coords[:, 1], s=1, alpha=0.4,
                           color=PALETTE["accent"], rasterized=True)
        else:
            idx_g = self.umap_glycan_combo.currentIndex()
            if idx_g < len(self.peaks):
                pk = self.peaks[idx_g]
                col_idx = self._get_peak_index(pk)
                if col_idx is not None:
                    v = np.asarray(adata.X, dtype=np.float32)[:, col_idx]
                    sc = ax.scatter(coords[:, 0], coords[:, 1], c=v, cmap="inferno",
                                    s=1, alpha=0.6,
                                    vmin=np.percentile(v, 1), vmax=np.percentile(v, 99),
                                    rasterized=True)
                    self.umap_canvas.fig.colorbar(sc, ax=ax, shrink=0.8, pad=0.01)
                    ax.set_title(
                        f"UMAP — {self.glycan_names[idx_g].split('(')[0][:20]}", fontsize=8.5
                    )
        ax.set_xlabel("UMAP 1", fontsize=8)
        ax.set_ylabel("UMAP 2", fontsize=8)
        self.umap_canvas.fig.tight_layout(pad=0.5)
        self.umap_canvas.draw()

    # ── Heatmap tab ───────────────────────────────────────────────────────

    def _build_heatmap_tab(self) -> QWidget:
        w = QWidget()
        layout = QVBoxLayout(w)
        layout.setContentsMargins(4, 4, 4, 4)

        ctrl_grp = QGroupBox("Options")
        ctrl_layout = QVBoxLayout(ctrl_grp)

        r1 = QHBoxLayout()
        r1.addWidget(QLabel("Group by:"))
        self.hmap_group_col = QComboBox()
        self._populate_obs_categoricals(self.hmap_group_col)
        r1.addWidget(self.hmap_group_col)
        ctrl_layout.addLayout(r1)

        r2 = QHBoxLayout()
        r2.addWidget(QLabel("Top N peaks:"))
        self.hmap_topn = QSpinBox()
        self.hmap_topn.setRange(5, 200)
        self.hmap_topn.setValue(30)
        r2.addWidget(self.hmap_topn)
        ctrl_layout.addLayout(r2)

        r3 = QHBoxLayout()
        r3.addWidget(QLabel("Normalise:"))
        self.hmap_norm_combo = QComboBox()
        self.hmap_norm_combo.addItems(["z-score", "min-max", "none"])
        r3.addWidget(self.hmap_norm_combo)
        ctrl_layout.addLayout(r3)

        layout.addWidget(ctrl_grp)
        plot_btn = QPushButton("Plot Heatmap")
        plot_btn.clicked.connect(self._draw_heatmap)
        layout.addWidget(plot_btn)
        self.hmap_canvas = MplCanvas(w, width=4.5, height=5.5, dpi=90)
        layout.addWidget(self.hmap_canvas)
        return w

    def _draw_heatmap(self):
        adata = self._adata()
        group_col = self.hmap_group_col.currentText()
        top_n = self.hmap_topn.value()
        norm = self.hmap_norm_combo.currentText()

        if group_col == "(none)" or group_col not in adata.obs.columns:
            show_info("Select a valid grouping column first.")
            return

        X = np.asarray(adata.X, dtype=np.float32)
        labels = adata.obs[group_col].astype(str).values
        groups = sorted(set(labels))
        means = np.array([X[labels == g].mean(axis=0) for g in groups])
        top_idx = np.argsort(means.var(axis=0))[::-1][:top_n]
        means_top = means[:, top_idx]

        if norm == "z-score":
            from scipy.stats import zscore
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                means_top = np.nan_to_num(zscore(means_top, axis=0))
        elif norm == "min-max":
            mn = means_top.min(axis=0, keepdims=True)
            mx = means_top.max(axis=0, keepdims=True)
            with np.errstate(divide="ignore", invalid="ignore"):
                means_top = np.where(mx > mn, (means_top - mn) / (mx - mn), 0.0)

        disp = _resolve_var_display_labels(adata)
        col_labels = [disp[i][:12] for i in top_idx]

        self.hmap_canvas.fig.clear()
        ax = self.hmap_canvas.fig.add_subplot(111)
        self.hmap_canvas._style_ax(ax)
        im = ax.imshow(means_top, aspect="auto", cmap="RdBu_r", interpolation="nearest")
        self.hmap_canvas.fig.colorbar(im, ax=ax, shrink=0.8, pad=0.01,
                                      label=norm if norm != "none" else "intensity")
        ax.set_yticks(range(len(groups)))
        ax.set_yticklabels(groups, fontsize=7)
        ax.set_xticks(range(len(col_labels)))
        ax.set_xticklabels(col_labels, rotation=90, fontsize=5.5, ha="center")
        ax.set_title(f"Mean intensity by {group_col}", fontsize=8.5)
        self.hmap_canvas.fig.tight_layout(pad=0.5)
        self.hmap_canvas.draw()

    # ── Stats tab ─────────────────────────────────────────────────────────

    def _build_stats_tab(self) -> QWidget:
        w = QWidget()
        layout = QVBoxLayout(w)
        layout.setContentsMargins(4, 4, 4, 4)
        plot_btn = QPushButton("Refresh Statistics")
        plot_btn.clicked.connect(self._draw_stats)
        layout.addWidget(plot_btn)
        self.stats_canvas = MplCanvas(w, width=4.5, height=5.5, dpi=90)
        layout.addWidget(self.stats_canvas)
        QTimer.singleShot(400, self._draw_stats)
        return w

    def _draw_stats(self):
        adata = self._adata()
        X = np.asarray(adata.X, dtype=np.float32)
        self.stats_canvas.fig.clear()
        axes = self.stats_canvas.fig.subplots(2, 2)
        for ax in axes.flat:
            self.stats_canvas._style_ax(ax)

        axes[0, 0].hist(X.sum(axis=1), bins=50,
                        color=PALETTE["accent"], edgecolor="none", alpha=0.8)
        axes[0, 0].set_title("TIC distribution", fontsize=8)

        axes[0, 1].hist((X > 0).sum(axis=1), bins=40,
                        color=PALETTE["accent2"], edgecolor="none", alpha=0.8)
        axes[0, 1].set_title("Peaks / pixel", fontsize=8)

        axes[1, 0].hist((X > 0).mean(axis=0) * 100, bins=40,
                        color=PALETTE["success"], edgecolor="none", alpha=0.8)
        axes[1, 0].set_title("Peak frequency (%)", fontsize=8)

        cluster_drawn = False
        for col in ("GPCA_clusters", "leiden", "batch", "annotation"):
            if col in adata.obs.columns:
                cats = adata.obs[col].astype(str)
                counts = cats.value_counts().sort_index()
                cmap = plt.get_cmap("tab20", len(counts))
                axes[1, 1].bar(range(len(counts)), counts.values,
                               color=[cmap(i) for i in range(len(counts))],
                               edgecolor="none", alpha=0.85)
                axes[1, 1].set_xticks(range(len(counts)))
                axes[1, 1].set_xticklabels(counts.index, rotation=45, ha="right", fontsize=6)
                axes[1, 1].set_title(f"Cluster sizes ({col})", fontsize=8)
                cluster_drawn = True
                break
        if not cluster_drawn:
            axes[1, 1].text(0.5, 0.5, "No cluster\ncolumn found",
                            ha="center", va="center", transform=axes[1, 1].transAxes,
                            color=PALETTE["text_dim"])

        for ax in axes.flat:
            ax.tick_params(labelsize=6.5)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        self.stats_canvas.fig.suptitle(
            f"{adata.n_obs:,} pixels × {adata.n_vars:,} peaks",
            fontsize=8.5, color=PALETTE["text"], y=1.01,
        )
        self.stats_canvas.fig.tight_layout(pad=0.5)
        self.stats_canvas.draw()


# ════════════════════════════════════════════════════════════════════════════
# 3. MAIN LAUNCH FUNCTION
# ════════════════════════════════════════════════════════════════════════════

def launch_goatpy_gui(
    sdata: SpatialData,
    peaks: Optional[list[float]] = None,
    glycan_csv: Optional[str] = None,
    table_name: str = "maldi_adata",
    viewer: Optional[napari.Viewer] = None,
) -> napari.Viewer:
    """
    Launch the goatpy napari GUI.

    Parameters
    ----------
    sdata       : processed SpatialData object
    peaks       : curated peak list (uses adata.var_names if None)
    glycan_csv  : path to glycan CSV; falls back to bundled glycan_list.csv
    table_name  : AnnData table key
    viewer      : existing napari viewer (new one created if None)
    """

    # ── Resolve peaks ────────────────────────────────────────────────────
    if peaks is None:
        try:
            peaks = list(_resolve_mz_array(sdata.tables[table_name]))
        except Exception:
            peaks = []

    # ── Load glycan annotation CSV ───────────────────────────────────────
    glycan_df = None
    csv_path = glycan_csv
    if csv_path is None:
        try:
            import pkg_resources
            csv_path = pkg_resources.resource_filename("goatpy", "data/glycan_list.csv")
        except Exception:
            pass
    if csv_path:
        try:
            raw = pd.read_csv(csv_path)
            raw.columns = [c.strip().strip('"') for c in raw.columns]
            mz_col = next(
                (c for c in raw.columns if any(k in c.lower()
                 for k in ["mz", "m/z", "theoretical"])),
                raw.columns[0],
            )
            label_col = next(
                (c for c in raw.columns if any(k in c.lower()
                 for k in ["composition", "name", "label", "glycan"])),
                raw.columns[1] if len(raw.columns) > 1 else None,
            )
            glycan_df = pd.DataFrame({
                "mz":    pd.to_numeric(raw[mz_col], errors="coerce"),
                "label": raw[label_col].astype(str) if label_col else "",
            }).dropna(subset=["mz"])
        except Exception as e:
            print(f"[goatpy GUI] Could not load glycan CSV: {e}")

    # ── Create viewer ────────────────────────────────────────────────────
    if viewer is None:
        viewer = napari.Viewer(title="goatpy — Spatial Glycomics Analysis")

    # ── Add H&E / IHC image ──────────────────────────────────────────────
    for img_key in ("he_image", "ihc_image", "optical_image"):
        if img_key in sdata.images:
            try:
                img_data = sdata.images[img_key]
                try:
                    arr = img_data["scale0"].ds[
                        list(img_data["scale0"].ds.data_vars)[0]
                    ].values
                except Exception:
                    arr = np.array(img_data) if hasattr(img_data, "__array__") else None
                if arr is not None:
                    if arr.ndim == 4:
                        arr = arr[0]
                    if arr.ndim == 3 and arr.shape[0] in (3, 4):
                        arr = np.transpose(arr, (1, 2, 0))
                    viewer.add_image(arr, name=img_key, rgb=(arr.ndim == 3))
            except Exception as e:
                print(f"[goatpy GUI] Could not add {img_key}: {e}")
            break

    # ── Apply Qt dark styles ──────────────────────────────────────────────
    app = QApplication.instance()
    if app:
        app.setStyleSheet(BASE_STYLE)

    # ── Spectrum widget ───────────────────────────────────────────────────
    spectrum_widget = SpectrumWidget(
        sdata=sdata, peaks=peaks, glycan_df=glycan_df, table_name=table_name
    )
    spectrum_widget.setStyleSheet(BASE_STYLE)
    spectrum_widget.setMinimumHeight(240)
    viewer.window.add_dock_widget(spectrum_widget, area="bottom", name="Spectrum")

    # ── Analysis sidebar ──────────────────────────────────────────────────
    sidebar = AnalysisSidebar(
        sdata=sdata, peaks=peaks, viewer=viewer,
        glycan_df=glycan_df, table_name=table_name,
    )
    sidebar.setStyleSheet(BASE_STYLE)
    sidebar.setMinimumWidth(340)
    sidebar.setMaximumWidth(460)
    viewer.window.add_dock_widget(sidebar, area="right", name="Analysis")

    # ── Cross-wiring ──────────────────────────────────────────────────────
    # Sidebar "Show on H&E" → highlight spectrum peak
    sidebar.glycan_selected.connect(
        lambda mz, lbl: spectrum_widget.highlight_glycan(mz, lbl)
    )
    # Spectrum click → select in sidebar + render on viewer
    spectrum_widget.peak_clicked.connect(sidebar.select_peak_from_spectrum)

    n_px = len(sdata.tables[table_name])
    print(
        f"[goatpy GUI] Ready.\n"
        f"  {n_px:,} pixels · {len(peaks):,} peaks\n"
        f"  Click a red peak in the spectrum  → renders ion map on H&E viewer\n"
        f"  Sidebar 'Show on H&E Viewer'      → same\n"
        f"  Spectrum: scroll=pan, Ctrl+scroll=zoom"
    )
    return viewer


# ════════════════════════════════════════════════════════════════════════════
# Standalone test with dummy data
# ════════════════════════════════════════════════════════════════════════════

def _make_dummy_sdata() -> SpatialData:
    import anndata as ad
    import geopandas as gpd
    from shapely.geometry import box
    from spatialdata.models import TableModel, ShapesModel, PointsModel
    from spatialdata.transformations import Identity

    np.random.seed(42)
    n, n_peaks = 400, 40
    peaks_mz = np.sort(np.random.uniform(900, 2800, n_peaks))
    X = np.abs(np.random.randn(n, n_peaks)).astype(np.float32)
    he_x = np.random.uniform(0, 1000, n)
    he_y = np.random.uniform(0, 800, n)

    obs = pd.DataFrame({
        "he_x": he_x, "he_y": he_y,
        "GPCA_clusters": pd.Categorical(np.random.choice(["0","1","2","3"], n)),
        "MPI": X.sum(axis=1),
    })
    adata = ad.AnnData(X=X, obs=obs)
    adata.var_names = [f"{m:.4f}" for m in peaks_mz]
    adata.obsm["spatial"] = obs[["he_x", "he_y"]].values
    adata.obsm["GraphPCA"] = np.random.randn(n, 10).astype(np.float32)

    geoms = [box(x-2, y-2, x+2, y+2) for x, y in zip(he_x, he_y)]
    gdf = gpd.GeoDataFrame({"cell_id": np.arange(n).astype(str)}, geometry=geoms)
    shapes = ShapesModel.parse(gdf, transformations={"global": Identity()})
    pts = pd.DataFrame({"x": he_x, "y": he_y, "cell_id": np.arange(n).astype(str)})
    centroids = PointsModel.parse(pts)

    sdata = SpatialData(shapes={"pixels": shapes}, points={"centroids": centroids})
    adata.obs["instance_id"] = gdf.index
    adata.obs["region"] = pd.Categorical(["pixels"] * n)
    table = TableModel.parse(adata, region="pixels",
                             region_key="region", instance_key="instance_id")
    sdata["maldi_adata"] = table

    # Add a dummy H&E image (1000×800 pink noise)
    from spatialdata.models import Image2DModel
    he = np.random.randint(200, 255, (3, 800, 1000), dtype=np.uint8)
    sdata["he_image"] = Image2DModel.parse(
        he, dims=("c","y","x"), transformations={"global": Identity()}
    )
    return sdata



In [28]:
sdata

SpatialData object, with associated Zarr store: /Users/andrewcauser/Documents/Griffith/test_data/ovarian_sample.zarr
├── Images
│     ├── 'he_image': DataArray[cyx] (3, 19160, 23633)
│     └── 'optical_image': DataTree[cyx] (3, 37200, 18000), (3, 18600, 9000), (3, 9300, 4500), (3, 4650, 2250)
├── Points
│     ├── 'centroids': DataFrame with shape: (<Delayed>, 112) (2D points)
│     ├── 'he_landmarks': DataFrame with shape: (<Delayed>, 2) (2D points)
│     └── 'maldi_landmarks': DataFrame with shape: (<Delayed>, 2) (2D points)
├── Shapes
│     └── 'pixels': GeoDataFrame shape: (16455, 2) (2D shapes)
└── Tables
      └── 'maldi_adata': AnnData (16455, 108)
with coordinate systems:
    ▸ 'aligned', with elements:
        he_image (Images), optical_image (Images), centroids (Points), he_landmarks (Points), maldi_landmarks (Points), pixels (Shapes)
    ▸ 'global', with elements:
        he_image (Images), optical_image (Images), centroids (Points), he_landmarks (Points), maldi_landmarks (Po

In [3]:
from spatialdata import read_zarr
sdata = read_zarr("/Users/andrewcauser/Documents/Griffith/test_data/ovarian_sample.zarr")

version mismatch: detected: RasterFormatV02, requested: FormatV04
/Users/andrewcauser/anaconda3/envs/maldi_env/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
/Users/andrewcauser/anaconda3/envs/maldi_env/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/andrewcauser/anaconda3/envs/maldi_env/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/andrewcauser/anaconda3/envs/maldi_env/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill

In [ ]:
viewer = launch_goatpy_gui(sdata)

2026-06-06 15:42:26.520 | WARNING  | napari_spatialdata._viewer:__init__:57 - Due to Shift-L being used as shortcut in napari, it is being deprecated and might not link a new layer to an existing SpatialData object in the viewer. Please use ⌘-L on MacOS or else Ctrl-L.


AttributeError: 'Interactive' object has no attribute 'viewer'

2026-06-06 15:42:35.855 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
/Users/andrewcauser/anaconda3/envs/maldi_env/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/Users/andrewcauser/anaconda3/envs/maldi_env/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (19160, 23633) exceeds GL_MAX_TEXTURE_SIZE 16384 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
2026-06-06 15:42:41.470 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-06-06 15:42:41.474 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-06-06 15:42:41.475 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-06-06 15:42:48.241 | DEBUG    | napari_spatialdata._view:_on_layer_